# Day 6 Lab - Transfer learning: teach YOLO to find brain tumours

YOLO is an object detector (day 4): one look at the image → boxes + classes. The model we use was **pre-trained**
on COCO - 118 000 everyday photos with 80 classes (person, car, dog, donut, ...). It has never seen a brain scan.
Today we **re-train** it on 893 brain MRI/CT images - with a few lines of code:
- **Part 1 - The COCO model as it is** (about 5 min): what does it see in a brain scan?
- **Part 2 - The data** (about 5 min): images and boxes
- **Part 3 - Fine-tune YOLO** (about 15 min): one line of code
- **Part 4 - Is pre-training worth it?** (about 20 min): fine-tuning vs frozen backbone vs from scratch
- **⭐ Bonus** (homework): more epochs, a bigger model

⚠️ **Runtime → Change runtime type → T4 GPU.** On a CPU one training run takes about 10 minutes instead of 1-2.

You cannot break anything - if something looks strange: *Runtime → Restart session*, then run the setup cells again.

## Setup - run these two cells first
The first cell installs **Ultralytics** (the YOLO library) and downloads the brain-tumour dataset (about 4 MB).

**Data:** 893 training and 223 validation images (512 × 512). Every image has a box around a region of the brain
with one of 2 classes: **negative** (no tumour) or **positive** (tumour). Dataset by Ultralytics, licence AGPL-3.0.

In [ ]:
!pip install -q ultralytics

In [ ]:
import glob

import matplotlib.pyplot as plt
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

data = check_det_dataset("brain-tumor.yaml")          # downloads the images once
VAL_IMAGES = sorted(glob.glob(data["val"] + "/*.jpg"))
print("classes:", data["names"], "  validation images:", len(VAL_IMAGES))

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `show_predictions(model)` - the model looks at 8 validation images; the boxes it finds are drawn in
- `show_labels()` - 8 training images with the boxes a doctor drew (the labels)
- `evaluate(model)` - tests a trained model on the 223 validation images and prints the most important numbers
- `plot_curves(...)` - draws mAP50 after every epoch for several trained models (learning curves)

In [ ]:
IMG_SIZE = 320       # images are resized to 320 x 320 - fast enough for Colab


def _grid(images, title):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, im in zip(axes.ravel(), images):
        ax.imshow(im); ax.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def show_predictions(model, title="predictions"):
    results = model.predict(VAL_IMAGES[:8], imgsz=IMG_SIZE, conf=0.25, verbose=False)
    for i, r in enumerate(results):
        found = [f"{model.names[int(c)]} {p:.2f}" for c, p in zip(r.boxes.cls, r.boxes.conf)]
        print(f"image {i + 1}:", ", ".join(found) if found else "nothing found")
    _grid([r.plot()[..., ::-1] for r in results], title)


def show_labels():
    import cv2
    images = []
    for path in sorted(glob.glob(data["train"] + "/*.jpg"))[:8]:
        im = cv2.imread(path)[..., ::-1].copy()
        h, w = im.shape[:2]
        for line in open(path.replace("images", "labels").replace(".jpg", ".txt")):
            c, x, y, bw, bh = map(float, line.split())
            color = (255, 80, 80) if c == 1 else (80, 160, 255)
            cv2.rectangle(im, (int((x - bw / 2) * w), int((y - bh / 2) * h)), (int((x + bw / 2) * w), int((y + bh / 2) * h)), color, 3)
            cv2.putText(im, data["names"][int(c)], (int((x - bw / 2) * w), int((y - bh / 2) * h) - 6), 0, 0.8, color, 2)
        images.append(im)
    _grid(images, "labels: red = positive (tumour), blue = negative")


def evaluate(model):
    if getattr(model, "trainer", None) is None:
        print("❌ this model is not trained yet - write the train line above and run the cell again")
        return {"mAP50": 0.0, "mAP50-95": 0.0}
    m = model.val(data="brain-tumor.yaml", imgsz=IMG_SIZE, verbose=False, plots=False)
    s = {"mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3),
         "precision": round(float(m.box.mp), 3), "recall": round(float(m.box.mr), 3)}
    print("✅", s)
    return s


def plot_curves(**models):
    import csv
    plt.figure(figsize=(9, 4))
    for name, model in models.items():
        if getattr(model, "trainer", None) is None:
            continue
        rows = list(csv.DictReader(open(f"{model.trainer.save_dir}/results.csv")))
        plt.plot([int(r["epoch"]) for r in rows], [float(r["metrics/mAP50(B)"]) for r in rows], "o-", label=name)
    plt.xlabel("epoch"); plt.ylabel("mAP50 (validation)"); plt.ylim(0, 0.7); plt.legend()
    plt.show()


print("helpers ready")

# Part 1 - The COCO model as it is (about 5 min)
`yolo11n.pt` = the smallest YOLO (2.6 million weights), pre-trained on COCO. Let it look at 8 brain scans.

In [ ]:
coco_model = YOLO("yolo11n.pt")
print("COCO classes:", len(coco_model.names), "- for example:", list(coco_model.names.values())[:10])
show_predictions(coco_model, "the COCO model - never saw a brain scan")

❓ What does the COCO model "see" in the brain scans? Why?

*Your answer:*

<details><summary>👀 What you should see</summary>

- in 5 of 8 images it finds something - **"donut"** (3 ×), **"apple"**, **"teddy bear"** - the other 3: nothing
- the boxes go around the **whole brain**: it can only answer with its 80 COCO classes, and a round object on a
  black background looks most like a donut to it
- but its backbone already knows **edges, round shapes and textures** - exactly what we need. We keep that and
  teach it the new classes
</details>

# Part 2 - The data (about 5 min)

In [ ]:
show_labels()

❓ How big are the boxes compared with the image? Could *you* tell positive from negative?

*Your answer:*

<details><summary>👀 What you should see</summary>

- most boxes are small - about 1/5 of the image width - around a bright or dark spot
- positive and negative regions can look similar for a layperson - a hard task with only 893 images
- training: 437 negative and 488 positive boxes; validation: 154 negative, 87 positive
</details>

# Part 3 - Fine-tune YOLO (about 15 min)
One line trains the model: start from the COCO weights, replace the head (80 → 2 classes) and train **all** layers
on the brain scans. Ultralytics prints a line per epoch and shows the validation scores at the end.

**mAP50** (mean average precision at IoU 0.5, day 4): 1.0 = every region found with a good box and the right class,
no extra boxes. 0 = nothing right.

In [ ]:
EPOCHS = 15          # 👈 about 1-2 min on a T4 GPU

finetuned = YOLO("yolo11n.pt")                                       # start: pre-trained on COCO
finetuned.train(data="brain-tumor.yaml", epochs=EPOCHS, imgsz=IMG_SIZE, seed=0, plots=False, verbose=False)
score_finetuned = evaluate(finetuned)
show_predictions(finetuned, "fine-tuned on brain scans")

❓ Compare with Part 1. What does the model find now? Which mistakes does it still make?

*Your answer:*

<details><summary>👀 What you should see</summary>

- after 15 epochs: **mAP50 about 0.47** (our CPU run) - and already **0.32 after the first epoch**
- the boxes now say "positive" / "negative" and sit on the bright spots - no more donuts
- typical mistakes: positive and negative confused, or two boxes for one region
- your numbers can differ a little (GPU vs CPU, random order of the images)
</details>

# Part 4 - Is pre-training worth it? (about 20 min)
The theory has three strategies (slides). Ultralytics needs only **one word** to switch:

| strategy | start weights | what is trained | code |
|---|---|---|---|
| from scratch | random | everything | `YOLO("yolo11n.yaml")` - `.yaml` = only the architecture, no weights |
| feature extraction | COCO | only the head - backbone **frozen** | `train(..., freeze=10)` - the first 10 layers stay fixed |
| fine-tuning | COCO | everything | Part 3 |

✏️ **Train the other two.** Copy the `train` line of Part 3 and change it.

In [ ]:
# Strategy: freeze the first 10 layers of the COCO model and train only the rest (neck + detection head) on the brain scans
frozen = YOLO("yolo11n.pt")
# TODO: Use the same train function as in Part (3) plus freeze=10
pass  # YOUR CODE HERE
score_frozen = evaluate(frozen)

In [ ]:
# Strategy: train a YOLO model from scratch on the brain scans (no weight initialization from COCO)
scratch = YOLO("yolo11n.yaml")                                       # random start: no pre-training
# TODO: Use the same train function as in Part (3) (without freeze=10)
pass  # YOUR CODE HERE
score_scratch = evaluate(scratch)

<details><summary>💡 Hint</summary>

- copy: `finetuned.train(data="brain-tumor.yaml", epochs=EPOCHS, imgsz=IMG_SIZE, seed=0, plots=False, verbose=False)`
- change `finetuned` to `frozen` (or `scratch`); for the frozen model add `, freeze=10` before the last `)`
- training takes 1-2 minutes per model on a T4 GPU - wait for the ✅ line
</details>

In [ ]:
print(f"{'strategy':20s} {'mAP50':>7s} {'mAP50-95':>9s}")
for name, s in [("from scratch", score_scratch), ("frozen backbone", score_frozen), ("fine-tuning", score_finetuned)]:
    print(f"{name:20s} {s['mAP50']:7.3f} {s['mAP50-95']:9.3f}")
plot_curves(**{"fine-tuning": finetuned, "frozen backbone": frozen, "from scratch": scratch})

❓ **Explain in two sentences:** look at the learning curves. What is the big difference in the **first 3 epochs** -
and what happens later?

*Your answer:*

<details><summary>👀 What you should see</summary>

Our run (CPU, 15 epochs, image size 320) - yours will be similar:

| strategy | mAP50 | mAP50-95 |
|---|---|---|
| strategy | mAP50 after epoch 1 | after epoch 3 | after epoch 15 | mAP50-95 (epoch 15) |
|---|---|---|---|---|
| from scratch | **0.00** | 0.18 | 0.47 | 0.31 |
| frozen backbone | 0.41 | 0.42 | **0.53** | 0.37 |
| fine-tuning | 0.32 | **0.46** | 0.47 | 0.33 |

- **head start**: the pre-trained models are good after 1-3 epochs; from scratch starts at 0 - the same network, but it
  must first learn edges and shapes from only 893 images. The COCO backbone brings them from 118 000 photos
- on this dataset from scratch catches up after about 10 epochs - the brain scans are a fairly easy, uniform task
- the frozen backbone wins here: with few images, training only the head overfits less (slide "Which strategy when?")
- the curves jump up and down by ±0.03 from epoch to epoch - do not over-interpret small differences (day 3: several
  seeds!). With less data the head start of pre-training matters more - try the bonus
</details>

# ⭐ Bonus (homework)
1. **More epochs:** `EPOCHS = 50` for the fine-tuned model - does mAP50 still go up?
2. **Less data:** add `fraction=0.2` to the `train` line (only 20 % of the training images) for fine-tuning and for
   from scratch. How big is the gap now?
3. **A bigger model:** `YOLO("yolo11s.pt")` (9.4 million weights instead of 2.6) - better? slower?
4. **Your own data:** [Roboflow Universe](https://universe.roboflow.com) has thousands of detection datasets in YOLO
   format - pick one and train on it with the same line.

# For your project
- **start from a pre-trained model** - it is (almost) always better than from scratch with little data
- **fine-tune everything** with a small learning rate when you have a few hundred images or more; freeze the backbone
  when you have very few images or little compute
- the input must look like the pre-training data: same image size range, same normalisation (mean/std of ImageNet)
- for the challenge (classification): `torchvision.models.resnet18(weights="DEFAULT")` or `timm` - replace the last
  layer by `nn.Linear(..., 7)` and train